In [ ]:
import json
import os
from collections import defaultdict

import io
import pandas as pd
import plotly.express as px
from pandas.api.types import infer_dtype

In [ ]:
import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
base_path = "./outputs_omer"
task = "mbpp"  # "gsm8k" or "humaneval" or "mbpp"
# base_path = "/share/kuleshov/yzs2/nvidia-collab"
# task = "humaneval.bak.bak"  # "gsm8k" or "humaneval" or "mbpp"
num_fewshot = {
    "gsm8k": 5,
    "humaneval": 0,
    "mbpp": 3,
}[task]
pass_at_1_str = {
    "gsm8k": "exact_match,flexible-extract",
    "humaneval": "pass@1,create_test",
    "mbpp": "pass_at_1,none",
}[task]

In [ ]:
def str2(string):
    try:
        return int(string)
    except ValueError:
        return bool(string)

result_dicts = []
for model, path in {
    "corrector": "corrector",
    # "corrector": "llada_8b_blend_code_math_resampling_v2_all_corrected_eps1e-1",
    # "instruct": "LLaDA-8B-Instruct",
}.items():
    model_path = os.path.join(base_path, path)
    results_dir = os.path.join(model_path, task, f"{num_fewshot=}")
    if not os.path.exists(results_dir):
        results_dir = os.path.join(model_path, task, f"num_fewshot-{num_fewshot}")
    for d in os.listdir(results_dir):
        result_dict = {"task": task, "model": model, "num_fewshot": num_fewshot} 
        if "=" in d:
            result_dict = result_dict | {p.split("=")[0]: str2(p.split("=")[1]) for p in d.split("-")}
        else:
            result_dict = result_dict | {p.split("-")[0]: str2(p.split("-")[1]) for p in d.split("--")}
        nfes = defaultdict(int)
        cnt = 0
        for dd in os.listdir(os.path.join(results_dir, d)):
            if "nfe" in dd:
                with open(os.path.join(results_dir, d, dd), "r") as f:
                    nfe_data = json.load(f)
                cnt += len(nfe_data["nfes_per_sequence"])
                for k, v in nfe_data["cumulative_nfes"].items():
                    nfes[k] += v
        
        result_dict = result_dict | {k: v / cnt if cnt != 0 else 0 for k, v in nfes.items()}
        try:
            for dd in os.listdir(os.path.join(results_dir, d, model_path.replace("/", "__"))):
                if "results" in dd:
                    with open(os.path.join(results_dir, d, model_path.replace("/", "__"), dd), "r") as f:
                        results_data = json.load(f)                    
                    result_dict["pass@1"] = results_data["results"][task.split(".")[0]]["pass_at_1_str"]
        except FileNotFoundError:
            for dd in os.listdir(os.path.join(results_dir, d)):
                if os.path.isdir(os.path.join(results_dir, d, dd)):                    
                    for ddd in os.listdir(os.path.join(results_dir, d, dd)):
                        if "results" in ddd:
                            with open(os.path.join(results_dir, d, dd, ddd), "r") as f:
                                results_data = json.load(f)
                            result_dict["pass@1"] = results_data["results"][task][pass_at_1_str]
        result_dicts.append(result_dict)

In [ ]:
df = pd.DataFrame.from_records(result_dicts)
df = df.dropna()
df.sort_values(
    by=["length", "steps", "model", "apply_corrector_every_n_steps", "max_corrector_steps_per_loop"]
)
# df

In [ ]:
filt_df = df.dropna()
sorted_categories = sorted(df['apply_corrector_every_n_steps'].unique())
# sorted_categories = sorted(df['apply_inner_corrector_every_n_steps'].unique())
fig = px.scatter(
    filt_df,
    x="total_nfe",
    y="pass@1",
    category_orders={"color": sorted_categories},
    color=filt_df["apply_corrector_every_n_steps"].astype(str),
    # color=filt_df["apply_inner_corrector_every_n_steps"].astype(str),
    color_discrete_sequence=px.colors.qualitative.Bold,
    size="max_corrector_steps_per_loop",
    # size="max_inner_corrector_steps_per_diffusion_step",    
    facet_col="steps",  # Creates the category split like a catplot
    labels={
        "total_nfe": "NFEs",
        "pass@1": "Accuracy (pass @ 1)",
        "apply_corrector_every_n_steps": "Apply Every N Steps",
        "max_corrector_steps_per_loop": "Max Corrector Steps per Loop"
        # "apply_inner_corrector_every_n_steps": "Apply Every N Steps",
        # "max_inner_corrector_steps_per_diffusion_step": "Max Corrector Steps per Loop"
    },
    hover_data={
        "apply_corrector_every_n_steps": True,
        "max_corrector_steps_per_loop": True,
        # "apply_inner_corrector_every_n_steps": True,
        # "max_inner_corrector_steps_per_diffusion_step": True,
        "total_nfe": True,
        "pass@1": True
    },
    title="Accuracy vs NFEs by Steps",
    size_max=24,
    width=1100,
    height=400,
)

# Show the plot
fig.show()